# Comprehensive Comparison

Side-by-side comparison of all methods across all experiments.  
Two aggregation views are available via `VIEW`:

| `VIEW` | Description |
|---|---|
| `"aggregated"` | Average over **all** noise configs (dims × scales × seeds) per env |
| `"canonical"` | Score at one representative `(noise_dim, noise_scale)` per env |

**Outputs per env**
- Ranked summary table (CSV)
- Grouped bar chart (by method category)
- Score heatmap (method × env)

**Cell layout**
- Cell 1 — Imports & project root  
- Cell 2 — Tunable config  
- Cell 3 — Plot style  
- Cell 4 — Method labels, groups, palette  
- Step 1 — Load all metrics  
- Step 2 — Aggregate scores  
- Step 3 — Save tables  
- Step 4 — Generate & save figures

In [ ]:
from pathlib import Path
import json
import re
import sys

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Cannot locate project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    RAW_METRICS_DIR,
    FIGURES_COMPREHENSIVE_DIR,
    TABLES_COMPREHENSIVE_DIR,
)

print(f"PROJECT_ROOT               = {PROJECT_ROOT}")
print(f"RAW_METRICS_DIR            = {RAW_METRICS_DIR}")
print(f"FIGURES_COMPREHENSIVE_DIR  = {FIGURES_COMPREHENSIVE_DIR}")
print(f"TABLES_COMPREHENSIVE_DIR   = {TABLES_COMPREHENSIVE_DIR}")

In [ ]:
# ============================================================
# Tunable config — modify this cell only
# ============================================================

# Aggregation view
#   "aggregated" — mean over all (dim, scale, seed) combinations
#   "canonical"  — score at one (dim, scale) per env (see ENV_CANONICAL below)
VIEW = "aggregated"

# Canonical noise config used when VIEW = "canonical"
# Chosen as the 0.5x dim / scale=1.0 representative point per env
ENV_CANONICAL = {
    "halfcheetah-medium-v2": {"dim": 8,  "scale": 1.0},
    "hopper-medium-v2":      {"dim": 6,  "scale": 1.0},
    "ant-medium-v2":         {"dim": 56, "scale": 1.0},
}

TARGET_ENVS  = [
    "halfcheetah-medium-v2",
    # "hopper-medium-v2",
    # "ant-medium-v2",
]
TARGET_SEEDS = [1, 2, 3]

# ── Methods to include ───────────────────────────────────────
# Comment out methods whose results are not yet available.
# Group key → list of raw_metrics method directory names.
METHOD_GROUPS = {
    "Main (IQL)": [
        "disentangled_barlow",
        "disentangled_dcor",
        "disentangled_hsic",
        "plain",
        "raw_noisy",
        "true_only",
    ],
    "Ablation: TD3+BC": [
        "disentangled_barlow_td3bc",
        "disentangled_dcor_td3bc",
        "disentangled_hsic_td3bc",
        "plain_td3bc",
        "true_only_td3bc",
    ],
    "Ablation: BC": [
        # "disentangled_barlow_bc",
        # "disentangled_dcor_bc",
        # "disentangled_hsic_bc",
        # "plain_bc",
        # "true_only_bc",
    ],
    "Ablation: Reward Only": [
        "disentangled_barlow_reward_only",
        "disentangled_dcor_reward_only",
        "disentangled_hsic_reward_only",
        "plain_reward_only",
    ],
    "Ablation: No Priv": [
        "disentangled_barlow_no_priv",
        "disentangled_dcor_no_priv",
        "disentangled_hsic_no_priv",
        "plain_no_priv",
    ],
    "External Methods": [
        # "pca_iql",
        # "riql",
        # "denoised_mdp",
    ],
}

# ── Noise config: auto-derived from env — DO NOT EDIT ────────
ENV_NOISE_DIMS = {
    "halfcheetah-medium-v2": [4, 8, 13, 17],
    "hopper-medium-v2":      [3, 6, 8, 11],
    "ant-medium-v2":         [28, 56, 83, 111],
}
NOISE_SCALES = [0.5, 1.0, 1.5, 2.0]
NOISE_TYPE   = "nonlinear"
# ─────────────────────────────────────────────────────────────

SAVE_FIGURES = True
SAVE_TABLES  = True
SHOW_FIGURES = True

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.1)

In [ ]:
# Display name for every possible method key
METHOD_LABELS = {
    # Main IQL
    "true_only":             "No Noise",
    "plain":                 "Plain",
    "raw_noisy":             "Raw Noisy",
    "disentangled_barlow":   "Barlow",
    "disentangled_dcor":     "dCor",
    "disentangled_hsic":     "HSIC",
    "disentangled_cov":      "Covariance",
    "disentangled_infonce":  "InfoNCE",
    "disentangled_l1":       "L1",
    # Algorithm ablations
    "true_only_td3bc":              "No Noise (TD3+BC)",
    "plain_td3bc":                  "Plain (TD3+BC)",
    "raw_noisy_td3bc":              "Raw Noisy (TD3+BC)",
    "disentangled_barlow_td3bc":    "Barlow (TD3+BC)",
    "disentangled_dcor_td3bc":      "dCor (TD3+BC)",
    "disentangled_hsic_td3bc":      "HSIC (TD3+BC)",
    "true_only_bc":                 "No Noise (BC)",
    "plain_bc":                     "Plain (BC)",
    "raw_noisy_bc":                 "Raw Noisy (BC)",
    "disentangled_barlow_bc":       "Barlow (BC)",
    "disentangled_dcor_bc":         "dCor (BC)",
    "disentangled_hsic_bc":         "HSIC (BC)",
    # Encoder ablations
    "plain_reward_only":                "Plain (Reward Only)",
    "disentangled_barlow_reward_only":   "Barlow (Reward Only)",
    "disentangled_dcor_reward_only":     "dCor (Reward Only)",
    "disentangled_hsic_reward_only":     "HSIC (Reward Only)",
    "plain_no_priv":                    "Plain (No Priv)",
    "disentangled_barlow_no_priv":       "Barlow (No Priv)",
    "disentangled_dcor_no_priv":         "dCor (No Priv)",
    "disentangled_hsic_no_priv":         "HSIC (No Priv)",
    # External
    "pca_iql":      "PCA-IQL",
    "riql":         "RIQL",
    "denoised_mdp": "Denoised MDP",
}

# Color per method group
GROUP_PALETTE = {
    "Main (IQL)":           "#2980b9",
    "Ablation: TD3+BC":     "#e74c3c",
    "Ablation: BC":         "#e67e22",
    "Ablation: Reward Only": "#8e44ad",
    "Ablation: No Priv":    "#27ae60",
    "External Methods":     "#16a085",
}

# Build flat method_key → group lookup
KEY_TO_GROUP = {
    key: grp
    for grp, keys in METHOD_GROUPS.items()
    for key in keys
}

# All active method keys (drop empty groups)
ALL_METHOD_KEYS = [
    k for keys in METHOD_GROUPS.values() for k in keys
]

print(f"Total methods configured : {len(ALL_METHOD_KEYS)}")
for grp, keys in METHOD_GROUPS.items():
    if keys:
        print(f"  {grp} ({len(keys)}): {keys}")

## Step 1 — Load all metrics

In [ ]:
def scale_to_tag(v: float) -> str:
    return str(v).replace(".", "p")


def _infer_method(norm: str, keys) -> str:
    # Longest match first to avoid "plain" matching "plain_td3bc"
    for k in sorted(keys, key=len, reverse=True):
        if f"/{k}/" in norm:
            return k
    return ""


def _infer_env(path: Path, envs) -> str:
    parts = set(path.parts)
    for e in envs:
        if e in parts:
            return e
    return ""


def load_all_metrics(
    raw_metrics_dir: Path,
    all_method_keys,
    target_seeds,
    target_envs,
    env_noise_dims,
    noise_scales,
    noise_type,
):
    """
    Returns
    -------
    df_noisy  : records with noise params
    df_oracle : records without noise params — true_only variants only
    """
    noisy, oracle = [], []
    seen = set()

    for fp in raw_metrics_dir.rglob("*.json"):
        key = str(fp.resolve())
        if key in seen:
            continue
        seen.add(key)

        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
        except Exception:
            continue

        score = data.get("normalized_score")
        if score is None:
            continue

        norm = str(fp.parent).replace("\\", "/")
        cfg  = data.get("data_config", {})

        method = (
            data.get("method") or data.get("group")
            or _infer_method(norm, all_method_keys)
        )
        if method not in all_method_keys:
            continue

        env = (
            data.get("env_name") or data.get("env")
            or cfg.get("env_name") or _infer_env(fp, target_envs)
        )
        if env not in target_envs:
            continue

        seed = data.get("seed", cfg.get("seed"))
        if seed is None:
            m = re.search(r"seed_?(\d+)", norm, re.IGNORECASE)
            seed = int(m.group(1)) if m else None
        else:
            seed = int(seed)
        if seed not in target_seeds:
            continue

        nd    = data.get("noise_dim",   cfg.get("noise_dim"))
        ns    = data.get("noise_scale", cfg.get("noise_scale"))
        ntype = data.get("noise_type",  cfg.get("noise_type"))

        # Oracle: only true_only variants lack noise params by design.
        # Other methods missing noise_type are legacy/old files — skip.
        if nd is None or ns is None or ntype is None:
            if method.startswith("true_only"):
                oracle.append({"Method": method, "Env": env,
                               "Seed": seed, "Score": float(score),
                               "Group": KEY_TO_GROUP.get(method, "Other")})
            continue

        if ntype != noise_type:
            continue
        env_dims = env_noise_dims.get(env, [])
        if int(nd) not in env_dims:
            continue
        if float(ns) not in noise_scales:
            continue

        noisy.append({
            "Method": method, "Env": env,
            "Dim": int(nd), "Scale": float(ns),
            "Type": ntype, "Seed": seed,
            "Score": float(score),
            "Group": KEY_TO_GROUP.get(method, "Other"),
        })

    df_noisy  = pd.DataFrame(noisy)
    df_oracle = (
        pd.DataFrame(oracle)
        .drop_duplicates(subset=["Method", "Env", "Seed"], keep="first")
        .reset_index(drop=True)
        if oracle else pd.DataFrame()
    )
    return df_noisy, df_oracle

In [ ]:
df_noisy, df_oracle = load_all_metrics(
    raw_metrics_dir=RAW_METRICS_DIR,
    all_method_keys=ALL_METHOD_KEYS,
    target_seeds=TARGET_SEEDS,
    target_envs=TARGET_ENVS,
    env_noise_dims=ENV_NOISE_DIMS,
    noise_scales=NOISE_SCALES,
    noise_type=NOISE_TYPE,
)

seed_tag = "_".join(str(s) for s in sorted(TARGET_SEEDS))

print(f"Noise-sweep records : {len(df_noisy)}")
print(f"Oracle records      : {len(df_oracle)}")
if not df_noisy.empty:
    print("\nMethods found in noise-sweep:")
    for grp, sub in df_noisy.groupby("Group"):
        print(f"  [{grp}] {sorted(sub['Method'].unique())}")
if not df_oracle.empty:
    print("\nOracle methods:", sorted(df_oracle['Method'].unique()))

## Step 2 — Aggregate scores

In [ ]:
def aggregate_scores(df_noisy, df_oracle, view, env_canonical):
    """
    Returns df_agg with columns:
      Method, Label, Group, Env, MeanScore, StdScore, N
    One row per (Method, Env).
    """
    rows = []

    # ── Noise-sweep methods ──
    if not df_noisy.empty:
        if view == "canonical":
            filtered = []
            for env, cfg in env_canonical.items():
                sub = df_noisy[
                    (df_noisy["Env"] == env)
                    & (df_noisy["Dim"] == cfg["dim"])
                    & (df_noisy["Scale"] == cfg["scale"])
                ]
                filtered.append(sub)
            work = pd.concat(filtered, ignore_index=True) if filtered else pd.DataFrame()
        else:  # aggregated
            work = df_noisy.copy()

        if not work.empty:
            for (method, env), grp in work.groupby(["Method", "Env"]):
                rows.append({
                    "Method":    method,
                    "Label":     METHOD_LABELS.get(method, method),
                    "Group":     KEY_TO_GROUP.get(method, "Other"),
                    "Env":       env,
                    "MeanScore": float(grp["Score"].mean()),
                    "StdScore":  float(grp["Score"].std(ddof=1)) if len(grp) > 1 else 0.0,
                    "N":         len(grp),
                })

    # ── Oracle methods (no noise params) ──
    if not df_oracle.empty:
        for (method, env), grp in df_oracle.groupby(["Method", "Env"]):
            rows.append({
                "Method":    method,
                "Label":     METHOD_LABELS.get(method, method),
                "Group":     KEY_TO_GROUP.get(method, "Other"),
                "Env":       env,
                "MeanScore": float(grp["Score"].mean()),
                "StdScore":  float(grp["Score"].std(ddof=1)) if len(grp) > 1 else 0.0,
                "N":         len(grp),
            })

    return pd.DataFrame(rows).sort_values(["Group", "Method", "Env"]).reset_index(drop=True)

In [ ]:
df_agg = aggregate_scores(df_noisy, df_oracle, VIEW, ENV_CANONICAL)

print(f"Aggregated rows: {len(df_agg)}  (view='{VIEW}')")
if not df_agg.empty:
    print("\nTop 10 by MeanScore:")
    display(
        df_agg[["Label", "Group", "Env", "MeanScore", "StdScore", "N"]]
        .sort_values("MeanScore", ascending=False)
        .head(10)
        .to_string(index=False)
    )

## Step 3 — Save tables

In [ ]:
def build_pivot_table(df_agg):
    """Wide table: rows = methods, columns = envs, values = MeanScore ± StdScore."""
    if df_agg.empty:
        return pd.DataFrame()
    pivot = df_agg.pivot_table(
        index=["Group", "Method", "Label"],
        columns="Env",
        values="MeanScore",
        aggfunc="mean",
    ).reset_index()
    # Overall mean across all envs
    env_cols = [c for c in pivot.columns if c not in ("Group", "Method", "Label")]
    pivot["Overall Mean"] = pivot[env_cols].mean(axis=1)
    return pivot.sort_values("Overall Mean", ascending=False).reset_index(drop=True)


def save_tables(df_agg, df_noisy, table_dir: Path, view, seed_tag):
    table_dir.mkdir(parents=True, exist_ok=True)

    if not df_agg.empty:
        p = table_dir / f"aggregated_{view}_seeds_{seed_tag}.csv"
        df_agg.to_csv(p, index=False)
        print(f"Saved aggregated → {p}")

        pivot = build_pivot_table(df_agg)
        p = table_dir / f"pivot_{view}_seeds_{seed_tag}.csv"
        pivot.to_csv(p, index=False)
        print(f"Saved pivot      → {p}")

    if not df_noisy.empty:
        p = table_dir / f"all_records_seeds_{seed_tag}.csv"
        df_noisy.to_csv(p, index=False)
        print(f"Saved raw        → {p}")

In [ ]:
if SAVE_TABLES:
    save_tables(df_agg, df_noisy, TABLES_COMPREHENSIVE_DIR, VIEW, seed_tag)
else:
    print("Table export disabled.")

## Step 4 — Generate & save figures

In [ ]:
def plot_comprehensive_bar(df_agg, env, view, GROUP_PALETTE, METHOD_LABELS,
                           save_path=None, show=True):
    """Grouped bar chart: all methods for one env, colored by group."""
    sub = df_agg[df_agg["Env"] == env].copy()
    if sub.empty:
        print(f"[WARN] No aggregated data for {env}")
        return

    sub = sub.sort_values(["Group", "MeanScore"], ascending=[True, False])
    colors = [GROUP_PALETTE.get(g, "#aaaaaa") for g in sub["Group"]]

    fig, ax = plt.subplots(figsize=(max(10, len(sub) * 0.7), 6))
    bars = ax.bar(
        range(len(sub)), sub["MeanScore"],
        yerr=sub["StdScore"], capsize=4,
        color=colors, edgecolor="black", linewidth=0.8,
        error_kw={"elinewidth": 1.2},
    )

    for bar, score in zip(bars, sub["MeanScore"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.5,
            f"{score:.1f}",
            ha="center", va="bottom", fontsize=8, fontweight="bold", rotation=90,
        )

    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["Label"], rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Normalized D4RL Score", fontsize=12)
    view_lbl = "Canonical" if view == "canonical" else "Aggregated (all noise configs)"
    ax.set_title(f"Comprehensive Comparison — {env}\n({view_lbl})",
                 fontsize=13, fontweight="bold", pad=12)

    scores = sub["MeanScore"][sub["MeanScore"] > 0]
    if not scores.empty:
        ax.set_ylim(max(0, scores.min() - 10), sub["MeanScore"].max() + 18)

    # Group legend
    from matplotlib.patches import Patch
    legend_handles = [
        Patch(facecolor=c, edgecolor="black", label=g)
        for g, c in GROUP_PALETTE.items()
        if g in sub["Group"].values
    ]
    ax.legend(handles=legend_handles, title="Category", fontsize=9,
              title_fontsize=10, loc="upper right")

    # Vertical separators between groups
    prev_group = None
    for i, grp in enumerate(sub["Group"]):
        if prev_group is not None and grp != prev_group:
            ax.axvline(i - 0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        prev_group = grp

    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show() if show else plt.close()


def plot_heatmap(df_agg, view, GROUP_PALETTE, save_path=None, show=True):
    """Heatmap: rows = methods (sorted by group), columns = envs."""
    if df_agg.empty:
        return

    pivot = df_agg.pivot_table(
        index=["Group", "Label"],
        columns="Env",
        values="MeanScore",
        aggfunc="mean",
    )
    pivot = pivot.sort_index(level="Group")

    n_rows, n_cols = pivot.shape
    fig, ax = plt.subplots(figsize=(max(5, n_cols * 2.5), max(6, n_rows * 0.5)))
    sns.heatmap(
        pivot, annot=True, fmt=".1f",
        cmap="YlOrRd", linewidths=0.5,
        ax=ax, cbar_kws={"label": "Normalized D4RL Score"},
    )

    view_lbl = "Canonical" if view == "canonical" else "Aggregated"
    ax.set_title(f"Score Heatmap — {view_lbl}",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Environment", fontsize=11)
    ax.set_ylabel("Method", fontsize=11)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right", fontsize=9)
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show() if show else plt.close()

In [ ]:
if df_agg.empty:
    print("[WARN] No aggregated data — skipping figures.")
else:
    fig_dir = FIGURES_COMPREHENSIVE_DIR

    # Per-env bar charts
    for env in TARGET_ENVS:
        print(f"\n--- {env} ---")
        plot_comprehensive_bar(
            df_agg, env, VIEW, GROUP_PALETTE, METHOD_LABELS,
            save_path=fig_dir / env / f"bar_{VIEW}.png" if SAVE_FIGURES else None,
            show=SHOW_FIGURES,
        )

    # Cross-env heatmap (only meaningful when multiple envs selected)
    if len(TARGET_ENVS) > 1:
        print("\n--- Heatmap (all envs) ---")
        plot_heatmap(
            df_agg, VIEW, GROUP_PALETTE,
            save_path=fig_dir / f"heatmap_{VIEW}.png" if SAVE_FIGURES else None,
            show=SHOW_FIGURES,
        )
    else:
        print("\n[INFO] Heatmap requires ≥ 2 envs — select more in TARGET_ENVS.")